In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score

In [31]:
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Пути
DATA_DIR = 'data'
ARTIFACTS_DIR = 'artifacts'
FIGURES_DIR = os.path.join(ARTIFACTS_DIR, 'figures')
LABELS_DIR = os.path.join(ARTIFACTS_DIR, 'labels')

# Глобальные хранилища для артефактов
metrics_summary = {}
best_configs = {}

Вспомогательные функции (EDA, Метрики, Графики)

In [32]:
def save_figure(fig, filename):
    path = os.path.join(FIGURES_DIR, filename)
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)

In [33]:
def calculate_metrics(X, labels):
    # Если кластеров < 2 или все точки шум - метрики не считаем
    unique_labels = set(labels)
    if len(unique_labels) < 2:
        return None
    # Для DBSCAN обрабатываем шум (label -1)
    mask = labels != -1
    # Если после удаления шума осталось меньше 2 кластеров или меньше 2 точек
    if len(set(labels[mask])) < 2 or np.sum(mask) < 2:
        return None

    X_clean = X[mask]
    labels_clean = labels[mask]

    return {
        "silhouette": float(silhouette_score(X_clean, labels_clean)),
        "davies_bouldin": float(davies_bouldin_score(X_clean, labels_clean)),
        "calinski_harabasz": float(calinski_harabasz_score(X_clean, labels_clean)),
        "noise_ratio": float(np.sum(labels == -1) / len(labels)) # Доля шума
    }


In [34]:
def run_kmeans_pipeline(X, dataset_name):
    print(f"Running KMeans for {dataset_name}")
    best_score = -1
    best_k = 2
    best_model = None
    best_labels = None
    
    silhouettes = []
    k_range = range(2, 11) # Пробуем от 2 до 10 кластеров
    
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X)
        score = silhouette_score(X, labels)
        silhouettes.append(score)
        
        if score > best_score:
            best_score = score
            best_k = k
            best_model = kmeans
            best_labels = labels
            
    # График Silhouette vs K
    fig, ax = plt.subplots()
    ax.plot(k_range, silhouettes, marker='o')
    ax.set_title(f'KMeans Silhouette vs K ({dataset_name})')
    ax.set_xlabel('Number of clusters (k)')
    ax.set_ylabel('Silhouette Score')
    save_figure(fig, f'{dataset_name}_kmeans_elbow.png')
    
    metrics = calculate_metrics(X, best_labels)
    metrics['noise_ratio'] = 0.0 # KMeans не имеет шума
    
    return {
        "model_name": "KMeans",
        "params": {"n_clusters": best_k},
        "metrics": metrics,
        "labels": best_labels
    }

In [35]:
def run_dbscan_pipeline(X, dataset_name):
    print(f"--- Running DBSCAN for {dataset_name} ---")
    # небольшой GridSearch
    eps_values = [0.1, 0.3, 0.5, 0.7, 1.0, 1.5]
    min_samples_values = [3, 5, 10]
    
    best_score = -1
    best_params = {}
    best_labels = None
    eps_plot_scores = []
    
    for eps in eps_values:
        for ms in min_samples_values:
            dbscan = DBSCAN(eps=eps, min_samples=ms)
            labels = dbscan.fit_predict(X)
            # Считаем метрику, если есть кластеры
            metrics = calculate_metrics(X, labels)
            score = -1
            if metrics:
                score = metrics['silhouette']
            
            if ms == 5: # Собираем для графика
                eps_plot_scores.append(score if score else 0)

            if score > best_score:
                best_score = score
                best_params = {"eps": eps, "min_samples": ms}
                best_labels = labels

    # График Silhouette vs Eps (при min_samples=5)
    fig, ax = plt.subplots()
    ax.plot(eps_values, eps_plot_scores, marker='o', color='orange')
    ax.set_title(f'DBSCAN Silhouette vs Eps (min_samples=5) ({dataset_name})')
    ax.set_xlabel('Eps')
    ax.set_ylabel('Silhouette Score')
    save_figure(fig, f'{dataset_name}_dbscan_tuning.png')

    final_metrics = calculate_metrics(X, best_labels)
    
    return {
        "model_name": "DBSCAN",
        "params": best_params,
        "metrics": final_metrics,
        "labels": best_labels
    }

In [36]:
def visualize_best_solution(X, labels, dataset_name, algo_name):
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)
    
    fig, ax = plt.subplots()
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='viridis', s=15, alpha=0.7)
    plt.colorbar(scatter)
    ax.set_title(f'PCA 2D: {dataset_name} | Best Algo: {algo_name}')
    save_figure(fig, f'{dataset_name}_best_pca.png')

Основной цикл обработки датасетов

In [37]:
datasets_to_process = [
    'S07-hw-dataset-01.csv', 
    'S07-hw-dataset-02.csv', 
    'S07-hw-dataset-03.csv'
]

for filename in datasets_to_process:
    ds_name = filename.split('.')[0]
    filepath = os.path.join(DATA_DIR, filename)
    
    print(f"\n{'='*20} Processing {ds_name} {'='*20}")
    
    # 1. Загрузка
    df = pd.read_csv(filepath)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    
    sample_ids = df['sample_id']
    X_raw = df.drop(columns=['sample_id'])

    print(df.info())
    
    # 2. Препроцессинг
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    X_scaled = pipeline.fit_transform(X_raw)
    
    # 3. Моделирование
    # 3.1 KMeans
    kmeans_res = run_kmeans_pipeline(X_scaled, ds_name)
    
    # 3.2 DBSCAN
    dbscan_res = run_dbscan_pipeline(X_scaled, ds_name)
    
    # 4. Сравнение и выбор лучшего
    km_sil = kmeans_res['metrics']['silhouette']
    db_sil = dbscan_res['metrics']['silhouette'] if dbscan_res['metrics'] else -1
    
    if db_sil > km_sil:
        winner = dbscan_res
    else:
        winner = kmeans_res
        
    print(f"Победитель для {ds_name}: {winner['model_name']} (Sil: {winner['metrics']['silhouette']:.3f})")
    
    # 5. Сохранение результатов (Артефакты)
    
    # Сохраняем Labels для победителя
    labels_df = pd.DataFrame({
        'sample_id': sample_ids,
        'cluster_label': winner['labels']
    })
    labels_df.to_csv(os.path.join(LABELS_DIR, f'labels_{ds_name}.csv'), index=False)
    
    # Визуализация победителя
    visualize_best_solution(X_scaled, winner['labels'], ds_name, winner['model_name'])
    
    # Запись в сводку
    metrics_summary[ds_name] = {
        "KMeans": kmeans_res['metrics'],
        "DBSCAN": dbscan_res['metrics']
    }
    
    best_configs[ds_name] = {
        "algorithm": winner['model_name'],
        "params": winner['params'],
        "reason": "Highest Silhouette Score (automated choice)"
    }



==================== Processing S07-hw-dataset-01 ====================
Shape: (12000, 9)
Columns: ['sample_id', 'f01', 'f02', 'f03', 'f04', 'f05', 'f06', 'f07', 'f08']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sample_id  12000 non-null  int64  
 1   f01        12000 non-null  float64
 2   f02        12000 non-null  float64
 3   f03        12000 non-null  float64
 4   f04        12000 non-null  float64
 5   f05        12000 non-null  float64
 6   f06        12000 non-null  float64
 7   f07        12000 non-null  float64
 8   f08        12000 non-null  float64
dtypes: float64(8), int64(1)
memory usage: 843.9 KB
None
Running KMeans for S07-hw-dataset-01
--- Running DBSCAN for S07-hw-dataset-01 ---
Победитель для S07-hw-dataset-01: KMeans (Sil: 0.522)

==================== Processing S07-hw-dataset-02 ====================
Shape: (8000, 4)
C

Проверка устойчивости (для Dataset-01)

In [38]:
print(f"\n{'='*20} Stability Check (Dataset 01) {'='*20}")

ds1_path = os.path.join(DATA_DIR, 'S07-hw-dataset-01.csv')
df1 = pd.read_csv(ds1_path)
X1 = df1.drop(columns=['sample_id'])
X1_scaled = StandardScaler().fit_transform(X1)

n_runs = 5
labels_runs = []
aris = []

print("Пробег KMeans 5 раз с разными seeds...")
for i in range(n_runs):
    km = KMeans(n_clusters=3, random_state=i*42, n_init=10)
    lbls = km.fit_predict(X1_scaled)
    labels_runs.append(lbls)
base_run = labels_runs[0]
for i in range(1, n_runs):
    ari = adjusted_rand_score(base_run, labels_runs[i])
    aris.append(ari)
    print(f"Run 0 vs Run {i}: ARI = {ari:.4f}")

avg_ari = np.mean(aris)
print(f"Средний ARI: {avg_ari:.4f}")
if avg_ari > 0.9:
    print("Заключение: Всё стабильно👍.")
else:
    print("Заключение: увы, тряска⚰️.")


==================== Stability Check (Dataset 01) ====================
Пробег KMeans 5 раз с разными seeds...
Run 0 vs Run 1: ARI = 1.0000
Run 0 vs Run 2: ARI = 1.0000
Run 0 vs Run 3: ARI = 1.0000
Run 0 vs Run 4: ARI = 1.0000
Средний ARI: 1.0000
Заключение: Всё стабильно👍.


In [39]:
# JSON артефакты
with open(os.path.join(ARTIFACTS_DIR, 'metrics_summary.json'), 'w') as f:
    json.dump(metrics_summary, f, indent=4)

with open(os.path.join(ARTIFACTS_DIR, 'best_configs.json'), 'w') as f:
    json.dump(best_configs, f, indent=4)

print("артефакты сохранены")

артефакты сохранены
